# Handwriting Analysis - CNN Model Training

This notebook trains a MobileNetV2-based CNN to classify handwriting samples
as **Normal**, **Reversal**, or **Corrected** for dyslexia screening.

## Dataset
Download the Kaggle Dyslexia Handwriting Dataset and place it in `../data/handwriting/`
with subdirectories: `Normal/`, `Reversal/`, `Corrected/`

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

import sys
sys.path.insert(0, os.path.abspath('..'))

from app.ml.handwriting.preprocessor import preprocess_image
from app.ml.handwriting.cnn_model import build_handwriting_model, fine_tune_model

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Configuration
DATA_DIR = '../data/handwriting'
CLASS_NAMES = ['Normal', 'Reversal', 'Corrected']
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10  # Train head only
EPOCHS_PHASE2 = 15  # Fine-tune
MODEL_SAVE_PATH = '../saved_models/handwriting_model.keras'

In [ ]:
# Load and preprocess dataset
images = []
labels = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATA_DIR, class_name)
    if not os.path.exists(class_dir):
        print(f'WARNING: {class_dir} not found!')
        continue
    
    files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f'{class_name}: {len(files)} images')
    
    for fname in files:
        fpath = os.path.join(class_dir, fname)
        try:
            img = preprocess_image(fpath)
            images.append(img)
            labels.append(class_idx)
        except Exception as e:
            print(f'Error processing {fpath}: {e}')

X = np.array(images)
y = keras.utils.to_categorical(labels, num_classes=len(CLASS_NAMES))
print(f'\nTotal: {len(X)} images, shape: {X.shape}')

In [ ]:
# Train/validation/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

In [ ]:
# Data augmentation
data_augmentation = keras.Sequential([
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomTranslation(0.05, 0.05),
])

In [ ]:
# Phase 1: Train classification head (base frozen)
model = build_handwriting_model(num_classes=len(CLASS_NAMES))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]

history1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE1,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
)

In [ ]:
# Phase 2: Fine-tune last 30 layers of MobileNetV2
model = fine_tune_model(model, unfreeze_layers=30)

history2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE2,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
)

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history2.history['accuracy'], label='Train')
ax1.plot(history2.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history2.history['loss'], label='Train')
ax2.plot(history2.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save model
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
model.save(MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')

In [ ]:
# Confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()